<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/04_train_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Mounted at /content/drive
Thiết bị tính toán đang dùng: cuda


In [2]:
# ===== CELL 2: NẠP TOÀN BỘ DỮ LIỆU TRỰC TIẾP LÊN VRAM GPU (SIÊU TỐC) =====
import numpy as np  # Import NumPy để đọc các file mảng .npy từ Google Drive

# Nạp trực tiếp dữ liệu tập Train lên bộ nhớ GPU (device='cuda') để loại bỏ hoàn toàn độ trễ truyền từ CPU
X_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy')), dtype=torch.float32, device=device)
Y_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy')), dtype=torch.float32, device=device)
S_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Steady_train.npy')), dtype=torch.float32, device=device)

# Nạp trực tiếp dữ liệu tập Validation lên bộ nhớ GPU
X_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy')), dtype=torch.float32, device=device)
Y_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy')), dtype=torch.float32, device=device)

# Nạp trực tiếp dữ liệu tập Test lên bộ nhớ GPU
X_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy')), dtype=torch.float32, device=device)
Y_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy')), dtype=torch.float32, device=device)

print(f'Đã nạp thẳng lên GPU ({device})! Số mẫu Train: {len(X_train_t)} | Số mẫu xác lập: {int(S_train_t.sum())}')

Đã nạp thẳng lên GPU (cuda)! Số mẫu Train: 89974 | Số mẫu xác lập: 63645


In [3]:
# ===== CELL 3: KIẾN TRÚC MẠNG PINN & PHYSICS LOSS (ĐÃ SỬA PSI_M & LỌC QUÁ ĐỘ) =====
import torch.nn as nn  # Import module mạng Neural của PyTorch
import joblib          # Import joblib để đọc bộ chuẩn hóa Scaler

# Nạp lại X_scaler và Y_scaler để giải chuẩn hóa dữ liệu về đơn vị vật lý (V, A, rpm, Ohm, H)
X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# Đưa giá trị trung bình (mean) và độ lệch chuẩn (scale) lên GPU dưới dạng Tensor
x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32).to(device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32).to(device)
y_mean = torch.tensor(Y_scaler.mean_, dtype=torch.float32).to(device)
y_scale = torch.tensor(Y_scaler.scale_, dtype=torch.float32).to(device)

class PMSM_PINN(nn.Module):
    """
    Kiến trúc mạng PINN gồm 5 đầu vào [ud, uq, Id, Iq, nm], 3 lớp ẩn 128 nút (Tanh)
    và 3 đầu ra ước lượng tham số [Rs, Ld, Lq].
    """
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_PINN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

def compute_physics_loss(inputs_scaled, outputs_scaled, steady_mask, pole_pairs=4, psi_m=0.527139):
    """
    Hàm tính sai số vật lý (Physics-informed Loss) dựa trên phương trình điện áp trạng thái xác lập của PMSM.
    Chỉ tính trên các mẫu thỏa mãn điều kiện xác lập (steady_mask == 1.0) và dùng đúng psi_m = 0.527139 Wb.
    """
    # Tạo mặt nạ lọc ra các mẫu đang ở trạng thái xác lập trong Batch
    mask = steady_mask > 0.5
    # Nếu cả Batch không có mẫu xác lập nào thì trả về loss vật lý bằng 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=inputs_scaled.device)

    # Chỉ giữ lại dữ liệu của các mẫu xác lập
    in_s = inputs_scaled[mask]
    out_s = outputs_scaled[mask]

    # Giải chuẩn hóa (Inverse StandardScaler): Giá_trị_thực = Giá_trị_chuẩn_hóa * Std + Mean
    inputs_real = in_s * x_scale + x_mean
    outputs_real = out_s * y_scale + y_mean

    # Tách các biến điện áp, dòng điện và tốc độ thực tế
    ud = inputs_real[:, 0]  # Điện áp trục d (V)
    uq = inputs_real[:, 1]  # Điện áp trục q (V)
    Id = inputs_real[:, 2]  # Dòng điện trục d (A)
    Iq = inputs_real[:, 3]  # Dòng điện trục q (A)
    nm = inputs_real[:, 4]  # Tốc độ cơ học (rpm)

    # Tách 3 tham số do mạng PINN dự đoán
    Rs_est = outputs_real[:, 0]  # Điện trở stator ước lượng (Ohm)
    Ld_est = outputs_real[:, 1]  # Điện cảm trục d ước lượng (H)
    Lq_est = outputs_real[:, 2]  # Điện cảm trục q ước lượng (H)

    # Quy đổi tốc độ cơ học nm (vòng/phút) sang tốc độ góc điện omega_e (rad/s)
    omega_m = nm * (2.0 * np.pi / 60.0)
    omega_e = pole_pairs * omega_m

    # Tính điện áp ud, uq lý thuyết theo phương trình trạng thái xác lập động cơ PMSM
    ud_calc = Rs_est * Id - omega_e * Lq_est * Iq
    uq_calc = Rs_est * Iq + omega_e * (Ld_est * Id + psi_m)

    # Tính sai số bình phương trung bình (MSE) giữa điện áp mô phỏng và điện áp vật lý tính được
    loss_ud = torch.mean((ud - ud_calc) ** 2)
    loss_uq = torch.mean((uq - uq_calc) ** 2)

    return loss_ud + loss_uq

# Khởi tạo mạng PINN và chuyển lên GPU
model_pinn = PMSM_PINN().to(device)
print("Đã khởi tạo mô hình PINN với psi_m = 0.527139 Wb trên thiết bị:", device)

Đã khởi tạo mô hình PINN với psi_m = 0.527139 Wb trên thiết bị: cuda


In [4]:
# ===== CELL 4: HUẤN LUYỆN PINN TRỰC TIẾP TRÊN GPU =====
import torch.optim as optim  # Import bộ tối ưu hóa trọng số Optimizer
import shutil                # Import thư viện shutil để copy file checkpoint sang Google Drive
import time                  # Import thư viện time để đo tổng thời gian huấn luyện

criterion_data = nn.MSELoss()                               # Hàm mất mát dữ liệu (Mean Squared Error)
optimizer = optim.Adam(model_pinn.parameters(), lr=0.002)   # Thuật toán tối ưu Adam với tốc độ học khởi tạo 0.002
# Bộ điều chỉnh tự động giảm một nửa Learning Rate nếu Val Loss không cải thiện sau 15 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

lambda_p = 1e-3              # Hệ số trọng số cân bằng thành phần Physics Loss trong tổng hàm Loss
epochs = 150                 # Tổng số vòng lặp huấn luyện toàn bộ tập dữ liệu
batch_size = 512             # Kích thước mỗi lô dữ liệu xử lý song song trên GPU
n_train = len(X_train_t)     # Tổng số mẫu trong tập Train
best_val_loss = float('inf') # Khởi tạo giá trị Val Loss tốt nhất ban đầu là vô cùng lớn

# Lưu file checkpoint tạm thời trên ổ cứng cục bộ Colab (/content) để không bị nghẽn mạng Drive
LOCAL_CKPT = '/content/best_pinn_model.pth'
DRIVE_CKPT = os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth')

print("=== BẮT ĐẦU HUẤN LUYỆN PINN (GPU-NATIVE) ===")
start_time = time.time()  # Bấm giờ bắt đầu huấn luyện

for epoch in range(1, epochs + 1):
    # --- GIAI ĐOẠN 1: HUẤN LUYỆN (TRAIN) ---
    model_pinn.train()  # Bật chế độ huấn luyện
    # Tạo hoán vị chỉ số ngẫu nhiên trực tiếp trên GPU để xáo trộn dữ liệu mỗi Epoch
    perm = torch.randperm(n_train, device=device)

    train_data_loss = 0.0
    train_phys_loss = 0.0

    # Duyệt qua từng Batch dữ liệu trực tiếp trên VRAM của GPU
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]  # Lấy danh sách chỉ số của Batch hiện tại
        inputs = X_train_t[idx]       # Lấy lô dữ liệu đầu vào [ud, uq, Id, Iq, nm]
        targets = Y_train_t[idx]      # Lấy lô nhãn thực tế [Rs, Ld, Lq]
        steady = S_train_t[idx]       # Lấy lô cờ trạng thái xác lập

        optimizer.zero_grad()           # Đặt lại gradient về 0 trước khi tính đạo hàm mới
        outputs = model_pinn(inputs)    # Lan truyền tiến: mạng PINN dự đoán [Rs, Ld, Lq]

        loss_d = criterion_data(outputs, targets)                  # Tính sai số dữ liệu (Data Loss)
        loss_p = compute_physics_loss(inputs, outputs, steady)     # Tính sai số vật lý (Physics Loss)
        loss_total = loss_d + lambda_p * loss_p                    # Tổng hợp hàm mất mát PINN

        loss_total.backward()           # Lan truyền ngược tính gradient
        optimizer.step()                # Cập nhật trọng số mạng

        # Cộng dồn giá trị Loss để thống kê trung bình cuối Epoch
        train_data_loss += loss_d.item() * len(idx)
        train_phys_loss += loss_p.item() * len(idx)

    train_data_loss /= n_train  # Tính Data Loss trung bình trên toàn tập Train
    train_phys_loss /= n_train  # Tính Phys Loss trung bình trên toàn tập Train

    # --- GIAI ĐOẠN 2: ĐÁNH GIÁ TRÊN VALIDATION ---
    model_pinn.eval()  # Chuyển mô hình sang chế độ đánh giá
    with torch.no_grad():  # Tắt cơ chế lưu đồ thị gradient để tiết kiệm bộ nhớ
        val_outputs = model_pinn(X_val_t)                   # Dự đoán toàn bộ tập Validation cùng lúc trên GPU
        val_loss = criterion_data(val_outputs, Y_val_t).item() # Tính sai số MSE trên tập Validation

    scheduler.step(val_loss)  # Cập nhật bộ điều chỉnh Learning Rate theo Val Loss

    # Lưu trọng số mô hình tốt nhất vào ổ cứng cục bộ /content
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pinn.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, LOCAL_CKPT)

    # In thông số huấn luyện và thời gian thực tế sau mỗi 15 epochs
    if epoch % 15 == 0 or epoch == 1:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch:3d}/{epochs}] | Data Loss: {train_data_loss:.6f} | Phys Loss: {train_phys_loss:.4f} | Val Loss: {val_loss:.6f} | Thời gian: {elapsed:.1f} giây")

# Copy file trọng số tốt nhất từ Colab sang Google Drive 1 lần duy nhất sau khi hoàn tất
shutil.copy(LOCAL_CKPT, DRIVE_CKPT)
print(f"\n--> Hoàn thành 150 epochs trong {time.time() - start_time:.1f} giây! Đã lưu vào: {DRIVE_CKPT}")

=== BẮT ĐẦU HUẤN LUYỆN PINN (GPU-NATIVE) ===
Epoch [  1/150] | Data Loss: 0.327631 | Phys Loss: 187.6702 | Val Loss: 0.578230 | Thời gian: 1.3 giây
Epoch [ 15/150] | Data Loss: 0.177413 | Phys Loss: 187.1863 | Val Loss: 0.704340 | Thời gian: 9.2 giây
Epoch [ 30/150] | Data Loss: 0.161361 | Phys Loss: 187.0849 | Val Loss: 0.711493 | Thời gian: 18.3 giây
Epoch [ 45/150] | Data Loss: 0.152289 | Phys Loss: 187.1735 | Val Loss: 0.685064 | Thời gian: 27.4 giây
Epoch [ 60/150] | Data Loss: 0.141344 | Phys Loss: 187.1503 | Val Loss: 0.738906 | Thời gian: 35.9 giây
Epoch [ 75/150] | Data Loss: 0.132163 | Phys Loss: 187.1894 | Val Loss: 0.646165 | Thời gian: 45.4 giây
Epoch [ 90/150] | Data Loss: 0.125732 | Phys Loss: 187.1347 | Val Loss: 0.649700 | Thời gian: 54.9 giây
Epoch [105/150] | Data Loss: 0.122051 | Phys Loss: 187.1840 | Val Loss: 0.645206 | Thời gian: 63.5 giây
Epoch [120/150] | Data Loss: 0.119927 | Phys Loss: 187.0833 | Val Loss: 0.651904 | Thời gian: 72.8 giây
Epoch [135/150] | Dat

In [5]:
# ===== CELL 5: ĐÁNH GIÁ PINN TRÊN TẬP TEST =====

# Nạp bản trọng số tốt nhất đã lưu trên Google Drive ra để đánh giá
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth'))
model_pinn.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trạng thái trọng số tối ưu
model_pinn.eval()                                          # Bật chế độ dự đoán

with torch.no_grad():  # Tắt tính toán gradient khi kiểm tra tập Test
    # Dự đoán toàn bộ tập Test trực tiếp trên GPU và chuyển kết quả về CPU dạng mảng NumPy
    all_preds = model_pinn(X_test_t).cpu().numpy()
    all_targets = Y_test_t.cpu().numpy()

# Giải chuẩn hóa (Inverse Transform) đưa kết quả về đơn vị vật lý thực (Ohm cho Rs, Henry cho Ld và Lq)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình (PINN): {mape[0]:.2f}%") # In % sai số của điện trở stator Rs
print(f"Sai số Ld trung bình (PINN): {mape[1]:.2f}%") # In % sai số của độ tự cảm trục d Ld
print(f"Sai số Lq trung bình (PINN): {mape[2]:.2f}%") # In % sai số của độ tự cảm trục q Lq

=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===
Sai số Rs trung bình (PINN): 6.21%
Sai số Ld trung bình (PINN): 0.06%
Sai số Lq trung bình (PINN): 0.16%
